# Enigma connector for Kaggle (dual T4) — Stokastisk 0.1.0 build 5

**You run this notebook yourself.** Nothing here starts automatically, and nothing opens an inbound port: the connector only makes *outbound HTTPS* calls to your Enigma API and talks to vLLM on `127.0.0.1`.

Use only within Kaggle's terms for **authorised personal experimentation**. Kaggle is not an always-on inference service; the session will end and Enigma will show the runtime as offline.

Settings: Accelerator **GPU T4 x2**, Internet **ON**, Python 3.13. Add a Kaggle **Secret** named `ENIGMA_RUNTIME_TOKEN` (create the runtime in Enigma with *Mock* turned **off**, then copy its key once). Attach the folder containing `enigma_connector/` as a Kaggle Dataset named `enigma-connector`.

Status: build 5 has **not** been run on a Kaggle GPU yet. Complete the checklist at the end and record the result in `docs/TEST_REPORT.md`.

## 1. Install Stokastisk (skip if already installed)
Pinned by URL and SHA256, PyPI as the only index, so Kaggle's torch/triton stay untouched (command copied from the SDK's INSTALL.md).

In [ ]:
!python -m pip install --index-url https://pypi.org/simple "stokastisk @ https://huggingface.co/waqasm86/stokastisk/resolve/main/wheels/stokastisk-0.1.0-5-cp313-cp313-linux_x86_64.whl#sha256=fe80d7c36fa78cf8a26feb1e00bb5f1f006ca45a027450353603976320ae0b3e"

## 2. Verify the environment (read-only)

In [ ]:
import sys, importlib.metadata as md
import pynvml
pynvml.nvmlInit()
n = pynvml.nvmlDeviceGetCount()
names = [pynvml.nvmlDeviceGetName(pynvml.nvmlDeviceGetHandleByIndex(i)) for i in range(n)]
pynvml.nvmlShutdown()
print("python", sys.version.split()[0], "| gpus:", n, names)
import stokastisk
print("stokastisk", stokastisk.__version__, "| prohibited_report clean:", stokastisk.prohibited_report()["clean"])
print("torch", md.version("torch"), "| triton", md.version("triton"), "| (these must be Kaggle's own pinned versions)")
assert n >= 1, "no GPU visible"

## 3. Make the connector importable (no pip changes to torch/triton)

In [ ]:
import sys, importlib
sys.path.insert(0, "/kaggle/input/enigma-connector")
for mod, dist in (("prometheus_client", "prometheus_client"), ("pynvml", "nvidia-ml-py")):
    try:
        importlib.import_module(mod)
    except ImportError:
        # --no-deps: add only this package, never resolve/upgrade anything else
        get_ipython().system(f"python -m pip install --no-deps {dist}")
import enigma_connector
print("enigma_connector", enigma_connector.__version__)

## 4. Credentials (from Kaggle Secrets; the key is never printed)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["ENIGMA_RUNTIME_TOKEN"] = UserSecretsClient().get_secret("ENIGMA_RUNTIME_TOKEN")
os.environ["ENIGMA_API_URL"] = "https://YOUR-ENIGMA-API-HOST"   # <- edit: your deployed Enigma API base URL (https)
os.environ["ENIGMA_MODE"] = "stokastisk"
os.environ["ENIGMA_VLLM_URL"] = "http://127.0.0.1:8000"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # the only model/mode with Kaggle evidence on the HTTP server path (TP=1, 2048 context)

## 5. Start vLLM through the SDK (explicit; TP=1, FP16, 2048 context)

In [ ]:
from stokastisk import StokastiskConfig
from stokastisk.serving import ServerProcess
server = ServerProcess(MODEL, StokastiskConfig(tensor_parallel_size=1, max_model_len=2048),
                       host="127.0.0.1", port=8000, served_model_name=MODEL)
server.start_and_wait(timeout=900)
print("healthy:", server.health(), "| log:", server.log_path)

## 6. Local smoke test (no Enigma involved)

In [ ]:
from stokastisk import serving
print("".join(serving.stream_chat(server.base_url, MODEL, [{"role": "user", "content": "Say hello in five words."}], max_tokens=24)))
m = serving.fetch_metrics(server.base_url)
print("vllm metric samples:", sum(k.startswith("vllm:") for k in m))

## 7. Connect to Enigma (outbound only)

In [ ]:
from enigma_connector.config import Config
from enigma_connector.cli import build
cfg = Config.from_env(mode="stokastisk", served_model=MODEL, tp=1, pp=1, max_model_len=2048)
connector = build(cfg)
connector.start()          # registers, then heartbeat + telemetry + job threads run in the background
print("connected as runtime", connector.runtime_id, "| run", connector.run_id)

Open Enigma → Playground and send a prompt. Re-run the next cell to watch counters.

In [ ]:
print(connector.stats, "| fatal:", connector.fatal, "| spooled batches:", len(connector.spool))

## 8. Shut down cleanly

In [ ]:
connector.stop()
if connector.gpu is not None:
    connector.gpu.close()
print("server exit code:", server.stop(), "| shutdown report:", server.shutdown_report)

## GPU acceptance checklist (tick each, then record in docs/TEST_REPORT.md)

- [ ] Step 2: 2 x Tesla T4 visible; torch/triton versions are Kaggle's own (unchanged by step 1)
- [ ] `stokastisk.prohibited_report()["clean"]` is `True`; wheel SHA256 matches RELEASE.json
- [ ] Step 5: `/health` returns 200 for Qwen2.5-0.5B-Instruct TP=1, 2048 context
- [ ] Step 6: streamed text is model-generated and `/metrics` has `vllm:` samples
- [ ] Step 7: runtime shows **Online** and **not mock** in Enigma
- [ ] A Playground prompt streams, persists after reload, and shows measured TTFT / latency
- [ ] GPU 0 and GPU 1 charts show real NVML values; unsupported counters show N/A
- [ ] vLLM page shows counters moving after the request; no negative rates
- [ ] Stop button halts generation mid-stream
- [ ] Step 8: both GPUs release memory; Enigma shows the runtime **Offline** within ~2 minutes
- [ ] Revoking the key in Enigma stops the connector (`connector.fatal` is set)

TP=2 / PP=2 HTTP serving and other models are **not** covered by this checklist and remain unverified.